In [4]:
import os
from dotenv import load_dotenv

In [5]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [6]:
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [7]:
from langchain_community.document_loaders import PyPDFLoader

In [8]:
cd d:/Projects/Question-Answer-Creator-Application/

d:\Projects\Question-Answer-Creator-Application


In [9]:
file_path = "data/statistics.pdf"
loader = PyPDFLoader(file_path)
data = loader.load()
data

[Document(metadata={'producer': 'Skia/PDF m120 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Report - Importance and the use of correlation in Statistics', 'source': 'data/statistics.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='ImportanceandtheuseofcorrelationinStatistics\nIntroduction\nCorrelationisastatistical measurethat expressestheextent towhichtwovariablesarelinearlyrelated. It isacommontool for describingsimplerelationshipswithout makingastatement about causeandeffect.Correlationcoefficientsrangefrom-1to1, withavalueof 0indicatingnolinear relationshipbetweenthetwovariables, avalueof 1indicatingaperfect positivelinear relationship, andavalueof -1indicatingaperfectnegativelinear relationship.\nCorrelationisimportantinstatisticsbecauseitcanbeusedto\n1. Identifyrelationshipsbetweenvariables: Correlationcanbeusedtoidentifywhether thereisarelationshipbetweentwovariables, andif so, whether therelationshipispositiveornegative. Thisinfor

In [10]:
content = ""
for page in data:
    content += page.page_content
content

'ImportanceandtheuseofcorrelationinStatistics\nIntroduction\nCorrelationisastatistical measurethat expressestheextent towhichtwovariablesarelinearlyrelated. It isacommontool for describingsimplerelationshipswithout makingastatement about causeandeffect.Correlationcoefficientsrangefrom-1to1, withavalueof 0indicatingnolinear relationshipbetweenthetwovariables, avalueof 1indicatingaperfect positivelinear relationship, andavalueof -1indicatingaperfectnegativelinear relationship.\nCorrelationisimportantinstatisticsbecauseitcanbeusedto\n1. Identifyrelationshipsbetweenvariables: Correlationcanbeusedtoidentifywhether thereisarelationshipbetweentwovariables, andif so, whether therelationshipispositiveornegative. Thisinformationcanbeuseful for understandingtherelationshipsbetweendifferent factorsinacomplexsystem.\n2. Makepredictions: If thereisastrongcorrelationbetweentwovariables, it ispossibletousethevalueof onevariabletopredictthevalueof theother variable. Thiscanbeuseful for makingprediction

In [11]:
from langchain_text_splitters import TokenTextSplitter
from transformers import AutoTokenizer

# Load a specific Hugging Face tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [12]:
# Create the splitter using the loaded tokenizer's encode method
text_splitter = TokenTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=200,
    chunk_overlap=50
)

In [13]:
# Split your content
chunks = text_splitter.split_text(content)

In [14]:
print(len(content),len(chunks))

3664 6


In [15]:
from langchain_core.documents import Document

# convert string chunks to Document format
document_chunks = [Document(page_content=chunk) for chunk in chunks]

In [16]:
document = Document(page_content=content)

In [17]:
from langchain_groq import ChatGroq

llamaChatModel = ChatGroq(model="llama-3.3-70b-versatile",temperature = 0.3)
openaiChatModel =  ChatGroq(model="openai/gpt-oss-120b",temperature = 0.3)
qwenChatModel =  ChatGroq(model="qwen/qwen3-32b",temperature = 0.3)

For question generation, we directly pass content to model. For Aswer generation, we store the content in vector database and do retrieval.

In [18]:
question_prompt_template = """
You are an expert at creating questions based on given materials and documentation.
Your goal is to prepare a student for their exam.
You do this by asking questions about the text below:

------------
{text}
------------

Create questions that will prepare the students for their exams.
Make sure not to loose any important information.
Respond with only questions. No other information.

QUESTIONS:
"""

In [19]:
from langchain_core.prompts import PromptTemplate

question_prompt = PromptTemplate(template = question_prompt_template, input_variables = ["text"])

In [20]:
refine_question_template = """
You are an expert at creating questions based on materials and documentation.
Your goal is to help a student to prepare for their exam.
We have recieved some questions to a certain extent : 
{existing_answer}

We have the option to refine the existing questions or add the new ones (only if necessary) with some more context below.

------------
{text}
------------

Given the new context, refine the original questions in English.
If the context is not helpful, please provided the original question.
Respond with only questions. No other information.

QUESTIONS:
"""

In [21]:
refine_question_prompt = PromptTemplate(
    input_variables = ["existing_answer","text"],
    template = refine_question_template
)

## Legacy chain

In [ ]:
from langchain_classic.chains.summarize import load_summarize_chain

question_chain = load_summarize_chain(
    llm = openaiChatModel,
    chain_type = "refine",
    verbose = True,
    question_prompt = question_prompt,
    refine_prompt = refine_question_prompt
)

questions = question_chain.run([document])
print(questions)

## LECL

In [22]:
from langchain_core.output_parsers import StrOutputParser

# Define the chains using LCEL
initial_chain = question_prompt | openaiChatModel | StrOutputParser()

refine_chain = refine_question_prompt | openaiChatModel | StrOutputParser()

print("Generating questions using LCEL pattern...")

# Start with the first chunk
current_questions = initial_chain.invoke({"text": document_chunks[0].page_content})

# Iteratively refine with the rest of the chunks
for i, doc in enumerate(document_chunks[1:], start=2):
    print(f"Refining with chunk {i}/{len(document_chunks)}...")
    current_questions = refine_chain.invoke({
        "existing_answer": current_questions,
        "text": doc.page_content
    })

print("\nFinal Questions Generated!")
print(current_questions)

Generating questions using LCEL pattern...
Refining with chunk 2/6...
Refining with chunk 3/6...
Refining with chunk 4/6...
Refining with chunk 5/6...
Refining with chunk 6/6...

Final Questions Generated!
1. What is correlation in statistics, and what type of relationship does it measure between two variables such as advertising spending and sales?  

2. How does a correlation coefficient differ from a claim of cause‑and‑effect, and why can even a very high correlation (e.g., between smoking rates and lung‑cancer incidence) not prove causation?  

3. What is the possible range of values for a correlation coefficient, and what do the extreme values (‑1, 0, +1) indicate about the direction and strength of the relationship (for example, between a stock’s return and a market index)?  

4. What does a correlation coefficient of 0 tell us about the linear relationship between two variables, such as anxiety scores and depression scores?  

5. What does a correlation coefficient of +1 indicat

In [23]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [24]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(document_chunks,embedding_model)

In [29]:
questions = current_questions.split("\n\n")
questions

['1. What is correlation in statistics, and what type of relationship does it measure between two variables such as advertising spending and sales?  ',
 '2. How does a correlation coefficient differ from a claim of cause‑and‑effect, and why can even a very high correlation (e.g., between smoking rates and lung‑cancer incidence) not prove causation?  ',
 '3. What is the possible range of values for a correlation coefficient, and what do the extreme values (‑1,\u202f0,\u202f+1) indicate about the direction and strength of the relationship (for example, between a stock’s return and a market index)?  ',
 '4. What does a correlation coefficient of\u202f0 tell us about the linear relationship between two variables, such as anxiety scores and depression scores?  ',
 '5. What does a correlation coefficient of\u202f+1 indicate regarding the direction, strength, and linearity of the relationship between variables like advertising budget and sales revenue?  ',
 '6. What does a correlation coeffic

## Legacy chain

In [30]:
from langchain_classic.chains import RetrievalQA

answer_chain = RetrievalQA.from_chain_type(
    llm = openaiChatModel,
    chain_type = "stuff",
    retriever = vector_store.as_retriever()
)

## LCEL

In [33]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [39]:
from langchain_core.runnables import RunnablePassthrough

prompt = PromptTemplate.from_template(
    """Answer the question based only on the context below:

{context}

Question: {question}
"""
)

retriever = vector_store.as_retriever()

answer_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | openaiChatModel
    | StrOutputParser()
)

In [ ]:
# Answer each question and save to a file

for question in questions:
    print(question)
    answer = answer_chain.invoke(question)
    print("Answer:",answer)
    print("------------------------------")
    with open("answers.txt","a") as file:
        file.write(question+"\n")
        file.write("Answer:"+answer+"\n")
        file.write("-------------------------------\n")

1. What is correlation in statistics, and what type of relationship does it measure between two variables such as advertising spending and sales?  
Answer: **Correlation** in statistics is a numerical measure that quantifies how strongly two variables move together in a *linear* way.  
It is expressed by a correlation coefficient that ranges from ‑1 to +1:

* **+1** – perfect positive linear relationship (as one variable increases, the other increases proportionally).  
* **‑1** – perfect negative linear relationship (as one variable increases, the other decreases proportionally).  
* **0** – no linear relationship.

Thus, when we look at **advertising spending and sales**, correlation tells us whether higher (or lower) advertising spending tends to be associated with higher (or lower) sales, and how strong that linear association is. It does not by itself prove causation, only the direction and strength of the linear relationship.
------------------------------
2. How does a correlati